<a href="https://colab.research.google.com/github/AntonioAgostini/MNLP-HW1/blob/main/HW1_MNLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Imports section**


**TODO**
- [?]  compute aggregated metric (e.g. mean(mrr, hit@k))
- [A5] Implement a reranker
- [A1] Use a differente similarity metrics
- [A3] use other baseline models yet implementd
- [B3]
- [B4]

**DONE**
- [B1]
- [B2]
- [A2]


In [ ]:
import torch
import torch.nn.functional as F
import pprint
import numpy as np
import pandas as pd
from typing import Dict
from random import choice
from datasets import load_dataset
from transformers import EarlyStoppingCallback
from sentence_transformers.evaluation import SentenceEvaluator
from sentence_transformers import (SentenceTransformer,
                                   SentenceTransformerTrainer,
                                   SentenceTransformerTrainingArguments,
                                   losses)

# **Constants**

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Training arguments
lr = 1e-5
epochs = 10
weight_decay = 0.001
training_bs = 32

Device: cuda


# **Data preprocessing**

In [ ]:
# Preparing pairs (anchor, positive)
def prepare_pair(example):
    return {
        'anchor': example['query'],
        'positive': example['answer']
    }

# Preparing triplets (anchor, positive, negative)
def prepare_triplet(example):
    negatives = [c for c in example["candidate_chunks"] if c != example["answer"]]
    neg = choice(negatives)
    return {
        "anchor": example["query"],
        "positive": example["answer"],
        "negative": neg
    }

ds = load_dataset("sapienzanlp-course-materials/hw-mnlp-2026")

# Splitting the given dataset in 90/10 train/dev (seed = 42 for reproducibility)
train_dev_split = ds['train'].train_test_split(test_size=0.1, seed=42)
ds_train = train_dev_split['train']
ds_dev = train_dev_split['test']
ds_test = ds['test']

pair_ds_train = ds_train.map(prepare_pair, remove_columns=ds_train.column_names)
pair_ds_dev = ds_dev.map(prepare_pair, remove_columns=ds_dev.column_names)

triplet_ds_train = ds_train.map(prepare_triplet, remove_columns=ds_train.column_names)
triplet_ds_dev = ds_dev.map(prepare_triplet, remove_columns=ds_dev.column_names)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/16.7M [00:00<?, ?B/s]

data/blind-00000-of-00001.parquet:   0%|          | 0.00/10.9M [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/66.6M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Map:   0%|          | 0/7200 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

# **Evaluation Metrics**

## **Helpers**

In [ ]:
def embed_example(_tuple, model):
    """
    Embed a single example using the given model. *Not used in the current implementation*.
    """
    # Embedding of queries
    q_embs = model.encode(
        _tuple["query"],
        show_progress_bar = False
    )

    # Embedding of the candidates
    cand_embs = model.encode(
        _tuple["candidate_chunks"],
        show_progress_bar = False
    )

    return {
        "q_emb": q_embs,
        "cand_embs": cand_embs,
        "answer_pos": _tuple["answer_pos"]
    }

def embed_batch(batch, model, batch_size=64):
    """
    Embed a batch of examples using the given model. Done to improve encoding speed.
    """
    q_embs = model.encode(
        batch["query"],
        batch_size=batch_size,
    )

    lengths = batch["n_candidates"]
    flat_candidates = []
    for cands in batch["candidate_chunks"]:
        for chunk in cands:
            flat_candidates.append(chunk)

    flat_cand_embs = model.encode(
        flat_candidates,
        batch_size=batch_size,
    )

    cand_embs = []
    start = 0
    for n in lengths:
        cand_embs.append(flat_cand_embs[start:start + n])
        start += n

    return {
        "q_emb": [emb for emb in q_embs],
        "cand_embs": cand_embs,
        "answer_pos": batch["answer_pos"],
    }

def preprocess_data(model, dataset, split_name = None, batch_size = 64):
    """
    Prepare an embedded version of the dataset
    """
    if split_name:
        dataset = dataset[split_name]

    emb_dataset = dataset.map(
        #embed_example, # To embed one example at time, enable this
        embed_batch,
        batched = True,
        batch_size = batch_size,
        remove_columns = dataset.column_names,
        fn_kwargs = {"model": model, "batch_size": batch_size},
    )

    return emb_dataset

def get_ranks(emb_dataset):
    """
    Compute the ranks of the correct answers for all queries in the dataset.
    Returns a list of integers [r_1, ..., r_n], where r_i is the rank of the
    correct answer for query `i` after sorting its candidate chunks by similarity score.
    """
    ranks = []

    for example in emb_dataset:
        q_emb = torch.as_tensor(example["q_emb"], dtype=torch.float32)
        cands_emb = torch.as_tensor(example["cand_embs"], dtype=torch.float32)
        answer_pos = int(example["answer_pos"])

        scores = F.cosine_similarity(q_emb, cands_emb)

        ranked_idx = torch.argsort(scores, descending=True)
        rank = ranked_idx.tolist().index(answer_pos)  + 1

        ranks.append(rank)

    return ranks # Shape [N], N number of queries


### **Hit@k**

Hit@k is a metric used to evaluate the accuracy of retrieval systems. It measures whether the correct answer is present among the top k chunks recommended by the system. If the relevant item (the *answer*) is found within the top k results, it's considered a **hit**.

The formula for Hit@k is:

$$Hit@k = \frac{1}{N} \sum_{i = 1}^{N} \mathbb{I}[rank_i \leq k]$$

where:
- $\mathbb{I}$ is the indicator function;
- $k$ is the size of the top ranked window within which the correct answer is considered a hit;
- $rank_i$ is the rank of the first relevant chunk for the $i$-th query.

In [ ]:
def compute_hit_contribute(k, rank):
    if rank <= k:
        return 1.0
    return 0.0


def compute_hit_at_k(k, ranks):
    hits = []

    for rank in ranks:
        hit = compute_hit_contribute(k, rank)
        hits.append(hit)

    return float(np.mean(hits))

def compute_multiple_hit_at_k(k_s, ranks):
    values = {}

    for k in k_s:
        values[k] = compute_hit_at_k(k, ranks)

    return values


### **MRR**

This metric assesses the effectiveness of a information retrivial system by giving higher scores to systems that rank relevant items higher. It is calculated as the average of the reciprocal ranks of the answer for a set of queries.

The formula for MRR is:

$$MRR = \frac{1}{N} \sum_{i=1}^{N} \frac{1}{rank_i}$$

where:
- $N$ is the total number of queries.
- $rank_i$ is the rank of the first relevant chunk for the $i$-th query.

In [ ]:
def compute_reciprocal_rank(rank):
    return 1.0 / rank

def compute_mrr(ranks):
    reciprocal_ranks = []

    for rank in ranks:
        rr = compute_reciprocal_rank(rank)
        reciprocal_ranks.append(rr)

    return float(np.mean(reciprocal_ranks))

# **Baseline**

In [ ]:
bert_baseline_model = SentenceTransformer("distilbert-base-uncased")
bert_baseline_model.to(device)

MiniLM_baseline_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
MiniLM_baseline_model.to(device)

# Precompute embeddings with the baseline models
bert_emb_dataset = preprocess_data(
    model = bert_baseline_model,
    dataset = ds,
    split_name = "test",
)
print(f"Shape of bert_emb_dataset: {len(bert_emb_dataset)}")

MiniLM_emb_dataset = preprocess_data(
    model = MiniLM_baseline_model,
    dataset = ds,
    split_name = "test",
)
print(f"Shape of MiniLM_emb_dataset: {len(MiniLM_emb_dataset)}")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Shape of bert_emb_dataset: 2000


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Shape of MiniLM_emb_dataset: 2000


In [ ]:
# Retrieving ranks
bert_ranks = get_ranks(bert_emb_dataset)
MiniLM_ranks = get_ranks(MiniLM_emb_dataset)

# Compute MRR with the baseline models
bert_baseline_mrr = compute_mrr(
    ranks = bert_ranks,
)

MiniLM_baseline_mrr = compute_mrr(
    ranks = MiniLM_ranks,
)

# Compute hit@k with the baseline models
k_s = [1, 3, 5]

bert_baseline_hits_at_k = compute_multiple_hit_at_k(
    ranks = bert_ranks,
    k_s = k_s,
)

MiniLM_baseline_hits_at_k = compute_multiple_hit_at_k(
    ranks = MiniLM_ranks,
    k_s = k_s,
)

print("Baseline Model: distilbert-base-uncased")
print(f"Baseline MRR: {bert_baseline_mrr:.4f}")
print("Baseline Hit@k:")
print(
    f"k = 1: {bert_baseline_hits_at_k[1]:.4f} | "
    f"k = 3: {bert_baseline_hits_at_k[3]:.4f} | "
    f"k = 5: {bert_baseline_hits_at_k[5]:.4f}"
)

print("Baseline Model: all-MiniLM-L6-v2")
print(f"Baseline MRR: {MiniLM_baseline_mrr:.4f}")
print("Baseline Hit@k:")
print(
    f"k = 1: {MiniLM_baseline_hits_at_k[1]:.4f} | "
    f"k = 3: {MiniLM_baseline_hits_at_k[3]:.4f} | "
    f"k = 5: {MiniLM_baseline_hits_at_k[5]:.4f}"
)

Baseline Model: distilbert-base-uncased
Baseline MRR: 0.3657
Baseline Hit@k:
k = 1: 0.1765 | k = 3: 0.4435 | k = 5: 0.6045
Baseline Model: all-MiniLM-L6-v2
Baseline MRR: 0.6094
Baseline Hit@k:
k = 1: 0.4480 | k = 3: 0.7170 | k = 5: 0.8080


# **Evaluator**

The evaluator measures the performance of the finetuned model by computing:
- the $MRR$
- the $Hit@k$ with $k \in \{1, 3, 5\}$

In [ ]:
class Evaluator(SentenceEvaluator):

    # The batch size is only to improve speed in encoding of chunks
    def __init__(self, dataset, name = None):
        self.dataset = dataset
        self.name = name

    def __call__(self, model, output_path=None, epoch: int = -1, steps: int = -1):

        # Preprocess embeddings inside the evaluator to use the current model
        emb_dataset = preprocess_data(
            model = model,
            dataset = self.dataset,
        )

        ranks = get_ranks(emb_dataset)

        mrr = compute_mrr(
            ranks = ranks,
        )

        k_s = [1, 3, 5]
        hits_at_k = compute_multiple_hit_at_k(
            ranks = ranks,
            k_s = k_s,
        )

        metrics = {"eval_mrr": mrr}
        for k, hit_value in hits_at_k.items():
            metrics[f"eval_hit@{k}"] = hit_value

        return metrics

# **Finetuning using triplets**
The finetuning process using triplets involves training the `SentenceTransformer` model to embed sentences such that an 'anchor' (the `query`) sentence is similar in the embedding space to a 'positive' sentence (the `correct answer`) than it is to a 'negative' sentence (the `unrelated answer`). This is achieved through a `TripletLoss` function, which optimizes the model to maintain a certain margin between the anchor-positive and anchor-negative distances.

### **Triplet loss**

*TODO*

In [ ]:

# Instantiate the model
model = SentenceTransformer("distilbert-base-uncased")
model.to(device)

# Instantiate the evaluator
evaluator = Evaluator(ds_dev)

# Triplet loss for (anchor, positive, negative) samples
loss = losses.TripletLoss(
    model=model,
    distance_metric=losses.TripletDistanceMetric.COSINE,
    triplet_margin=0.25,
)

# Training arguments
training_args = SentenceTransformerTrainingArguments(
    output_dir = "weights",
    num_train_epochs = epochs,
    per_device_train_batch_size = training_bs,
    per_device_eval_batch_size = training_bs,
    learning_rate = lr,
    weight_decay = weight_decay,

    eval_strategy = "epoch",
    save_strategy = "epoch",
    logging_steps = 50,

    load_best_model_at_end=True,
    metric_for_best_model = "eval_hit@3",
    greater_is_better = True,

    warmup_steps = 0.1
)

# Instantiate the trainer
# Notice that the optimizer is not explicitly chosen. By default, AdamW is used.
trainer = SentenceTransformerTrainer(model = model,
                                     args = training_args,
                                     train_dataset = triplet_ds_train,
                                     eval_dataset = triplet_ds_dev,
                                     evaluator = evaluator,
                                     loss = loss,
                                     callbacks = [EarlyStoppingCallback(early_stopping_patience=2)]
                                     # Early stopping callback stops the execution if the choosen metric
                                     # doesn't improve after `early_stopping_patience` evaluation steps
                                     # (defined in the `eval_strategy` field).
                                     )

# Launching the training
trainer.train()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Epoch,Training Loss,Validation Loss,Mrr,Hit@1,Hit@3,Hit@5
1,0.065204,0.081164,0.667512,0.533750,0.760000,0.838750
2,0.045761,0.061804,0.698450,0.566250,0.785000,0.876250
3,0.025624,0.062464,0.704968,0.578750,0.792500,0.868750
4,0.013404,0.062354,0.700421,0.572500,0.791250,0.872500
5,0.005856,0.064080,0.701266,0.578750,0.786250,0.861250


Map:   0%|          | 0/800 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

TrainOutput(global_step=2250, training_loss=0.04591235399246216, metrics={'train_runtime': 6415.5363, 'train_samples_per_second': 11.223, 'train_steps_per_second': 0.701, 'total_flos': 0.0, 'train_loss': 0.04591235399246216, 'epoch': 5.0})